# TabFM vs GBM — 信用貸し倒れ検証（再現ノートブック）

Googleの **TabFM**（ゼロショットのテーブル基盤モデル）は、信用の貸し倒れ予測でよくチューニングしたGBMに勝てるでしょうか。公開データ（UCI 台湾クレジットカード貸し倒れ）で競わせた検証の再現コードです。

- 記事全文: **[han-co.com/ja/blog/tabfm-credit](https://han-co.com/ja/blog/tabfm-credit)**
- TabFM 原文: [Google Research ブログ](https://research.google/blog/introducing-tabfm-a-zero-shot-foundation-model-for-tabular-data/) · [GitHub](https://github.com/google-research/tabfm)

> **注意**
> - このノートブックは **モデル（TabFMの重み）を含みません。** 実行するとHugging Face(`google/tabfm-1.0.0-pytorch`)から自動でダウンロードします（VRAM 約6.5GB）。
> - TabFMのarmは **CUDA GPUが必要**です（CPUは10倍以上遅い）。以下は8GB GPU向けの設定で、大きいGPUならコンテキストを増やしてください。
> - GBMのarmだけ見るならTabFMのセルは飛ばして構いません。

主要な3つのarm（**素のGBM・チューニングGBM・TabFMゼロショット**）を示します。CatBoost・XGBoost・TabFMアンサンブルまで含む全6armはリポジトリの `src/`（config ベース）で実行します。


## 実行方法

Python 3.12 前提。仮想環境を作って導入します。

```bash
python -m venv .venv
# Windows: .\.venv\Scripts\Activate.ps1   /   mac·linux: source .venv/bin/activate
pip install -r requirements.txt
# TabFM（CUDA torch を先に、その後ソース導入）
pip install torch --index-url https://download.pytorch.org/whl/cu124
pip install "tabfm[pytorch] @ git+https://github.com/google-research/tabfm.git"
```

小ネタ: TabFMはGPUメモリを多く使うため、このノートブックは **データを先に読み込んでから** TabFMのセルで `torch`/`tabfm` を import します。


In [ ]:
# 최초 1회만 (주석 해제). 위 '실행 방법' 참고. TabFM arm은 GPU + CUDA 필요.
# %pip install -q numpy pandas scikit-learn lightgbm optuna ucimlrepo
# %pip install -q torch --index-url https://download.pytorch.org/whl/cu124
# %pip install -q "tabfm[pytorch] @ git+https://github.com/google-research/tabfm.git"


## 1. データ — UCI 台湾クレジットカード貸し倒れ

3万人、23特徴量、翌月の貸し倒れ有無（二値）。貸し倒れ率 約22%。`ucimlrepo` で自動取得します。列は位置基準で標準名を付け、ローダが違っても後続コードが常に同じ列を見つけられるようにします。


In [ ]:
import numpy as np, pandas as pd

CANONICAL = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
             "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
             *[f"BILL_AMT{i}" for i in range(1, 7)],
             *[f"PAY_AMT{i}"  for i in range(1, 7)]]
CATEG = ["SEX", "EDUCATION", "MARRIAGE"]

def load_uci():
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=350)
    X = ds.data.features.copy()
    X.columns = CANONICAL                                   # 위치 기준 표준화
    y = pd.to_numeric(ds.data.targets.iloc[:, 0]).astype(int).to_numpy()
    return X, y

X, y = load_uci()
print(X.shape, "| 부도율 =", round(float(y.mean()), 4))


## 2. 特徴量エンジニアリング（チューニングGBM用）

信用アナリストが作りそうなドメイン特徴量です。利用率、返済カバレッジ、延滞プロファイル・動態（連続延滞、直近 対 過去の状態変化 など）。**これがまさにTabFMの相手**です。ゼロショットモデルが、この手作り特徴量なしでどこまで迫るかを見ます。（TabFMには元の23特徴量だけ渡します。）


In [ ]:
def engineer_credit_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy(); eps = 1.0
    limit = X["LIMIT_BAL"].astype(float)
    BILL = [f"BILL_AMT{i}" for i in range(1, 7)]
    AMT  = [f"PAY_AMT{i}"  for i in range(1, 7)]
    PAY  = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

    # 이용률: 한도 대비 청구액
    for i, c in enumerate(BILL, 1):
        X[f"UTIL_{i}"] = X[c].astype(float) / (limit + eps)
    util = [f"UTIL_{i}" for i in range(1, 7)]
    X["UTIL_MEAN"], X["UTIL_MAX"], X["UTIL_STD"] = X[util].mean(1), X[util].max(1), X[util].std(1)
    X["LAST_UTIL"] = X["UTIL_1"]

    # 납부 커버리지: 청구 대비 납부
    for i in range(1, 6):
        X[f"PAYRATIO_{i}"] = X[f"PAY_AMT{i}"].astype(float) / (X[f"BILL_AMT{i+1}"].astype(float).abs() + eps)
    pr = [f"PAYRATIO_{i}" for i in range(1, 6)]
    X["PAYRATIO_MEAN"], X["PAYRATIO_MIN"] = X[pr].mean(1), X[pr].min(1)

    # 연체 프로필 (연체상태 컬럼)
    pay = X[PAY].astype(float)
    X["PAY_MAX"], X["PAY_MEAN"], X["PAY_SUM"] = pay.max(1), pay.mean(1), pay.sum(1)
    X["N_DELAYED"] = (pay > 0).sum(1)
    X["N_SEVERE_DELAY"] = (pay >= 2).sum(1)
    X["PAY_TREND"] = pay["PAY_0"] - pay["PAY_6"]

    # 청구·납부 규모/추세
    bill, amt = X[BILL].astype(float), X[AMT].astype(float)
    X["BILL_MEAN"], X["BILL_STD"] = bill.mean(1), bill.std(1)
    X["BILL_TREND"] = bill["BILL_AMT1"] - bill["BILL_AMT6"]
    X["AMT_MEAN"], X["AMT_SUM"], X["AMT_STD"] = amt.mean(1), amt.sum(1), amt.std(1)
    X["ZERO_PAYMENTS"] = (amt == 0).sum(1)

    # 연체 동태: 연속 연체 streak, 마지막 연체 이후 개월, 최근3 vs 과거3
    p = X[PAY].to_numpy(float); d = (p > 0).astype(int)
    streak = np.zeros(len(X)); cur = np.zeros(len(X))
    for j in range(d.shape[1]):
        cur = (cur + d[:, j]) * d[:, j]
        streak = np.maximum(streak, cur)
    X["DELINQ_STREAK_MAX"] = streak
    X["MONTHS_SINCE_DELINQ"] = np.where(d.any(1), d.argmax(1), 6)
    X["PAY_STATUS_RECENT3"] = pay[["PAY_0", "PAY_2", "PAY_3"]].mean(1)
    X["PAY_STATUS_OLD3"]    = pay[["PAY_4", "PAY_5", "PAY_6"]].mean(1)
    X["PAY_STATUS_DELTA"]   = X["PAY_STATUS_RECENT3"] - X["PAY_STATUS_OLD3"]
    X["WORSENING"] = (pay["PAY_0"] > pay["PAY_6"]).astype(int)

    # 총 상환율 / 한도 상대
    X["TOTAL_PAID_TO_BILLED"] = amt.sum(1) / (bill.abs().sum(1) + eps)
    X["LIMIT_PER_AGE"] = limit / (X["AGE"].astype(float) + eps)

    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    return X.fillna(0.0)

print("엔지니어링 후 피처 수:", engineer_credit_features(X).shape[1])


## 3. 評価指標 — 判別 + キャリブレーション

信用は順位（判別）だけでなく確率（キャリブレーション）も重要です。判別は **ROC-AUC・PR-AUC・KS**、キャリブレーションは **Brier・LogLoss・ECE** で見ます。PR-AUCは稀な貸し倒れをどれだけ捉えるか、ECEは予測確率が実際の貸し倒れ率とどれだけ合うかを見ます。


In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             brier_score_loss, log_loss, roc_curve)

def ks_statistic(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return float(np.max(tpr - fpr))

def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(y_prob, edges[1:-1]), 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b; cnt = int(m.sum())
        if cnt:
            ece += (cnt / len(y_true)) * abs(y_true[m].mean() - y_prob[m].mean())
    return float(ece)

def compute_metrics(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.clip(np.asarray(y_prob, float), 1e-7, 1 - 1e-7)
    auc = roc_auc_score(y_true, y_prob)
    return {"roc_auc": float(auc), "gini": float(2 * auc - 1),
            "pr_auc": float(average_precision_score(y_true, y_prob)),
            "ks": ks_statistic(y_true, y_prob),
            "brier": float(brier_score_loss(y_true, y_prob)),
            "log_loss": float(log_loss(y_true, y_prob)),
            "ece": expected_calibration_error(y_true, y_prob)}


## 4. モデル — 3つのarm

- **素のGBM**: LightGBMの既定値、特徴量・チューニングなし（労力の下限）
- **チューニングGBM**: エンジニアリング特徴量 + Optunaチューニング + early stopping（強いベースライン）
- **TabFMゼロショット**: 素のまま、チューニング0。Hugging Faceから読み込み、1回のforward passで予測

**公平比較の原則**: 確率が実際の貸し倒れ率と合うよう、`class_weight` を与えず自然な比率で学習します（キャリブレーションの公平比較）。判別指標はこの選択にほぼ不変です。

**8GB GPU 設定（TabFM）**: 重みだけで6.5GBなので、コンテキストを1,000行にキャップし、estimator 32個でそれぞれ別の1,000行を見せて学習セットを実質すべてカバーします（peak VRAMは同じ）。テスト予測は500行ずつ分割。**16GB以上なら** `CONTEXT_MAX` を増やしアンサンブルのプリセットを使ってください。


In [ ]:
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier, early_stopping
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def gbm_raw_proba(Xtr, ytr, Xte):
    for c in CATEG:
        Xtr[c] = Xtr[c].astype("category"); Xte[c] = Xte[c].astype("category")
    m = LGBMClassifier(n_estimators=300, learning_rate=0.05, objective="binary",
                       n_jobs=-1, verbosity=-1, random_state=42)
    m.fit(Xtr, ytr)
    return m.predict_proba(Xte)[:, 1]

OPTUNA_TRIALS = 40   # 블로그 실험은 80. 노트북에선 시간 고려해 40 (원하면 올리세요).

def gbm_full_proba(Xtr, ytr, Xte, seed=42):
    Xtr, Xte = engineer_credit_features(Xtr), engineer_credit_features(Xte)
    for c in CATEG:
        Xtr[c] = Xtr[c].astype("category"); Xte[c] = Xte[c].astype("category")
    inner = list(StratifiedKFold(3, shuffle=True, random_state=seed).split(Xtr, ytr))

    def objective(t):
        p = dict(learning_rate=t.suggest_float("learning_rate", 0.01, 0.2, log=True),
                 num_leaves=t.suggest_int("num_leaves", 15, 255),
                 max_depth=t.suggest_int("max_depth", 3, 12),
                 min_child_samples=t.suggest_int("min_child_samples", 5, 120),
                 subsample=t.suggest_float("subsample", 0.6, 1.0),
                 colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1.0),
                 reg_alpha=t.suggest_float("reg_alpha", 1e-8, 10, log=True),
                 reg_lambda=t.suggest_float("reg_lambda", 1e-8, 10, log=True))
        sc, it = [], []
        for tr, va in inner:
            m = LGBMClassifier(n_estimators=2500, objective="binary", n_jobs=-1,
                               verbosity=-1, random_state=seed, subsample_freq=1, **p)
            m.fit(Xtr.iloc[tr], ytr[tr], eval_set=[(Xtr.iloc[va], ytr[va])],
                  eval_metric="auc", callbacks=[early_stopping(80, verbose=False)])
            sc.append(roc_auc_score(ytr[va], m.predict_proba(Xtr.iloc[va])[:, 1]))
            it.append(m.best_iteration_ or 2500)
        t.set_user_attr("n_est", float(np.mean(it)))
        return float(np.mean(sc))

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=OPTUNA_TRIALS)
    n_est = max(50, min(2500, int(round(study.best_trial.user_attrs["n_est"] * 1.05))))
    m = LGBMClassifier(n_estimators=n_est, objective="binary", n_jobs=-1, verbosity=-1,
                       random_state=seed, subsample_freq=1, **study.best_params)
    m.fit(Xtr, ytr)
    return m.predict_proba(Xte)[:, 1]


In [ ]:
# --- TabFM 제로샷 (CUDA GPU 필요; 여기서 torch/tabfm를 import) ---
CONTEXT_MAX  = 1000     # 8GB 기준. 16GB+면 키우세요 (예: 12000).
N_ESTIMATORS = 32       # 논문 기본값. 각 estimator가 다른 1,000행을 봐 학습셋을 커버.
PREDICT_BATCH = 500

def as_tabfm_frame(X):
    X = X.copy()
    for c in X.columns:
        X[c] = X[c].astype(str) if c in CATEG else pd.to_numeric(X[c], errors="coerce").astype(float)
    return X

_TABFM = None
def _load_tabfm():
    global _TABFM
    if _TABFM is None:
        from tabfm import tabfm_v1_0_0_pytorch as T
        _TABFM = T.load(device="cuda")      # HF에서 가중치 다운로드 (~6.5GB VRAM)
    return _TABFM

def tabfm_zeroshot_proba(Xtr, ytr, Xte):
    from tabfm import TabFMClassifier
    clf = TabFMClassifier(model=_load_tabfm(), n_estimators=N_ESTIMATORS,
                          max_num_rows=CONTEXT_MAX, random_state=42, verbose=False)
    clf.fit(as_tabfm_frame(Xtr), ytr)
    Xte = as_tabfm_frame(Xte)
    parts = [clf.predict_proba(Xte.iloc[i:i + PREDICT_BATCH])[:, 1]   # 메모리 안전 배치 예측
             for i in range(0, len(Xte), PREDICT_BATCH)]
    return np.concatenate(parts)


## 5. 交差検証（層化5-fold）

同じ分割で3つのarmを回します。TabFMはGPUがないとこのセルで時間がかかるか失敗するので、GBMだけ見るなら `ARMS` から `tabfm_zeroshot` を外してください。


In [ ]:
import time

ARMS = {"gbm_raw": gbm_raw_proba, "gbm_full": gbm_full_proba,
        "tabfm_zeroshot": tabfm_zeroshot_proba}
# GPU가 없으면: ARMS.pop("tabfm_zeroshot")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, fn in ARMS.items():
    for k, (tr, te) in enumerate(skf.split(X, y)):
        t0 = time.time()
        p = fn(X.iloc[tr].copy(), y[tr], X.iloc[te].copy())
        m = compute_metrics(y[te], p)
        m.update(arm=name, fold=k, sec=round(time.time() - t0, 1))
        rows.append(m)
        print(f"{name:16s} fold{k}: AUC={m['roc_auc']:.4f}  KS={m['ks']:.3f}  ECE={m['ece']:.3f}")
res = pd.DataFrame(rows)


## 6. 結果


In [ ]:
order = [a for a in ["gbm_full", "tabfm_zeroshot", "gbm_raw"] if a in res["arm"].unique()]
summary = (res.groupby("arm")[["roc_auc", "pr_auc", "ks", "brier", "ece", "sec"]]
             .mean().round(4).reindex(order))
summary


## まとめ

- よくチューニングしたGBMがTabFMゼロショットをわずかに上回ります（ブログ実験基準で AUC 0.789 対 0.785、fold ノイズの範囲内）。
- 労力なし同士（素のGBM 対 TabFMゼロショット）ならTabFMが上です。
- キャリブレーションは互角です（`class_weight` なしで学習した場合）。

**全6arm**（CatBoost・XGBoost・TabFMアンサンブル含む）と正確な再現設定はリポジトリの `src/` と `config.yaml` にあります。**TabFMアンサンブルのプリセット**（cross+SVD特徴量 + NNLS重み + Plattキャリブレーション）は大きなコンテキストが要るため **16GB以上のGPU** を推奨します。

記事全文: **[han-co.com/ja/blog/tabfm-credit](https://han-co.com/ja/blog/tabfm-credit)**
